# Selection and generation of CROP-Seq library

- Refiltering of sensor editing data to ensure only clean edits/high efficiency gRNAs present
- Include truncating gRNA controls
- Include NT controls
- Limit size to 100-150
- Awareness of ABE vs. CBE (make libraries for both)

In [1]:
import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt 
import scipy.stats
import os
from adjustText import adjust_text
import matplotlib.patheffects as PathEffects
import warnings
warnings.filterwarnings('ignore')
plt.rc('font', family='Helvetica')

In [4]:
ABE_LFC = pd.read_csv('../../screening_data/03_LFC_FDR_tables/ABE_master_LFC_table.csv')
CBE_LFC = pd.read_csv('../../screening_data/03_LFC_FDR_tables/CBE_master_LFC_table.csv')



In [40]:
conditions_s1 = ['KI-CDK9d-32_100nM', 
                 'KI-CDK9d-32_1000nM', 
                 'KI-CDK9d-32N_1250nM', 
                 'KI-CDK9d-32N_5000nM', 
                 'KB-0742_1500nM', 
                 'Senexin B_2000nM', 
                 'Senexin B_15000nM',
                 'SEL120_4000nM',
                 'SY-5609_10nM',
                 'SY-5609_100nM',]

conditions_s2 = ['BSJ-4-116', 'CDK12-IN-2', 'HQ461']

conditions_s3 = ['ABEMA', 'ATIRMO', 'INX', 'PALBO', 'RIBO', 'TAGTO']

conditions_comb = conditions_s1 + conditions_s2 + conditions_s3


In [ ]:
ABE_LFC['base']

In [190]:
def generate_top_table(ABE_LFC, CBE_LFC, conditions_comb, LFC_cutoff, FDR_cutoff):
    atop = []
    ctop = []
    for cond in conditions_comb:
        abe_top = ABE_LFC[(ABE_LFC[f'LFC_median_{cond}_DMSO']>LFC_cutoff) & (ABE_LFC[f'FDR_{cond}_DMSO']<FDR_cutoff)].copy()
        abe_top2 = abe_top[['gRNA_id', 'classification', 'Screen_ID', 'Editor', f'LFC_median_{cond}_DMSO', f'FDR_{cond}_DMSO', 'base_RAW_DMSO']].rename(columns={f'LFC_median_{cond}_DMSO':'LFC', f'FDR_{cond}_DMSO':'FDR'})
        abe_top2['Condition_significant'] = cond
        atop.append(abe_top2)

        cbe_top = CBE_LFC[(CBE_LFC[f'LFC_median_{cond}_DMSO']>LFC_cutoff) & (CBE_LFC[f'FDR_{cond}_DMSO']<FDR_cutoff)].copy()
        cbe_top2 = cbe_top[['gRNA_id', 'classification', 'Screen_ID', 'Editor', f'LFC_median_{cond}_DMSO', f'FDR_{cond}_DMSO', 'base_RAW_DMSO']].rename(columns={f'LFC_median_{cond}_DMSO':'LFC', f'FDR_{cond}_DMSO':'FDR'})
        cbe_top2['Condition_significant'] = cond
        ctop.append(cbe_top2)   

    acomb = pd.concat(atop).reset_index(drop=True)
    ccomb = pd.concat(ctop).reset_index(drop=True)

    u, c = np.unique(acomb['gRNA_id'], return_counts=True)
    repeats_abe = pd.DataFrame({'gRNA_id':u, 'num_conditions_significant':c}).sort_values('num_conditions_significant', ascending=False)

    u, c = np.unique(ccomb['gRNA_id'], return_counts=True)
    repeats_cbe = pd.DataFrame({'gRNA_id':u, 'num_conditions_significant':c}).sort_values('num_conditions_significant', ascending=False)

    #----now add information about number of conditions with significant enrichment and remove duplicates, keeping condition with highest enrichment as LFC/FDR value

    m_abe = pd.merge(acomb, repeats_abe, on='gRNA_id')
    m_cbe = pd.merge(ccomb, repeats_cbe, on='gRNA_id')

    #remove duplicates, keeping condition with highest enrichment as LFC/FDR value
    m_abe = m_abe.sort_values(by='LFC', ascending=False).drop_duplicates(subset=['gRNA_id']).reset_index(drop=True)
    m_cbe = m_cbe.sort_values(by='LFC', ascending=False).drop_duplicates(subset=['gRNA_id']).reset_index(drop=True)

    #also add information about the T0 vs. plasmid LFC and FDR for each guide, to get a sense of variant basal fitness
    t0_subset_abe = ABE_LFC[(ABE_LFC['gRNA_id'].isin(m_abe['gRNA_id'])) & (ABE_LFC['Screen_ID'].isin(['Subpool1', 'Subpool2', 'Subpool3']))]
    t0_abe = t0_subset_abe[['gRNA_id', 'LFC_median_T0_Plasmid', 'FDR_T0_Plasmid']].dropna()

    t0_subset_cbe = CBE_LFC[(CBE_LFC['gRNA_id'].isin(m_cbe['gRNA_id'])) & (CBE_LFC['Screen_ID'].isin(['Subpool1', 'Subpool2', 'Subpool3']))]
    t0_cbe = t0_subset_cbe[['gRNA_id', 'LFC_median_T0_Plasmid', 'FDR_T0_Plasmid']].dropna()

    assert len(m_abe)==len(t0_abe)
    assert len(m_cbe)==len(t0_cbe)
    m_abe1 = pd.merge(m_abe, t0_abe, on='gRNA_id')
    m_cbe1 = pd.merge(m_cbe, t0_cbe, on='gRNA_id')

    return m_abe1, m_cbe1




In [199]:
LFC_cutoff = 1.5
FDR_cutoff = 0.01

m_abe, m_cbe = generate_top_table(ABE_LFC, CBE_LFC, conditions_comb, LFC_cutoff, FDR_cutoff)

print(len(m_abe))
print(len(m_cbe))

166
59


In [200]:
m_abe

,gRNA_id,classification,Screen_ID,Editor,LFC,FDR,base_RAW_DMSO,Condition_significant,num_conditions_significant,LFC_median_T0_Plasmid,FDR_T0_Plasmid
0,gRNA_CDK12_targ_2232,targeting,Subpool2,ABE,7.317829,9.985563e-07,3369.0,HQ461,3,-0.181775,0.233930
1,gRNA_CDK12_targ_2238,targeting,Subpool2,ABE,6.565316,9.985563e-07,563.0,HQ461,1,-2.123748,0.366900
2,gRNA_CDK13_targ_4604,targeting,Subpool2,ABE,4.873180,6.657042e-07,4449.0,BSJ-4-116,2,-0.623260,0.506805
3,gRNA_CDK7_targ_282,targeting,SY-5609,ABE,4.436557,4.731663e-03,1127.0,SY-5609_10nM,2,-0.586601,1.000000
4,gRNA_CDK12_targ_2285,targeting,Subpool2,ABE,4.365312,6.657042e-07,5265.0,BSJ-4-116,1,0.454953,0.277567
...,...,...,...,...,...,...,...,...,...,...,...
161,gRNA_CDK7_targ_352,targeting,Subpool1,ABE,1.512057,4.570356e-04,8675.0,KI-CDK9d-32_100nM,1,-0.157050,0.836914
162,gRNA_CDK9_targ_1439,targeting,Subpool1,ABE,1.509153,1.011503e-03,6224.0,KI-CDK9d-32_100nM,1,-0.475329,0.984520
163,gRNA_CDK8_intron_8541,intron,Subpool1,ABE,1.508004,4.570356e-04,3174.0,KI-CDK9d-32_100nM,1,-0.679284,1.000000
164,gRNA_CDK8_targ_675,targeting,Subpool1,ABE,1.506163,6.770765e-04,4920.0,KI-CDK9d-32_100nM,1,-0.410494,1.000000


# editing filtration and merging

- also add information about whether the amino acid falls in the KLIFS catalytic domain
- And add info about non-canonical editing percentage (don't include in adjusted %)

In [102]:
edict = {'ABE-CDK2_4_6':'ABE_CDK2_4_6_full.zip',
         'ABE-CDK12_13':'ABE_CDK12_13_full.zip',
         'ABE-CDK7_8_9':'ABE_subpool1_full.zip',
         'CBE-CDK2_4_6':'CBE_CDK2_4_6_full.zip',
         'CBE-CDK12_13':'CBE_CDK12_13_full.zip',
         'CBE-CDK7_8_9':'CBE_subpool1_full.zip'}

ah = []
for key in edict.keys():
    f = edict[key]
    a = pd.read_csv(f'../../screening_data/04_editing/{f}')
    a['Editor'] = key.split('-')[0]
    ah.append(a)

combined_editing = pd.concat(ah)
combined_editing['Gene'] = [x.split('_')[1] for x in combined_editing['gRNA_id']]

ABE_canonical_edits = combined_editing[(combined_editing['Canonical_edit']==True) & (combined_editing['Editor']=='ABE')].reset_index(drop=True)
CBE_canonical_edits = combined_editing[(combined_editing['Canonical_edit']==True) & (combined_editing['Editor']=='CBE')].reset_index(drop=True)

ABE_non_canonical_edits = combined_editing[(combined_editing['Canonical_edit']==False) & (combined_editing['Editor']=='ABE')].reset_index(drop=True)
CBE_non_canonical_edits = combined_editing[(combined_editing['Canonical_edit']==False) & (combined_editing['Editor']=='CBE')].reset_index(drop=True)

In [262]:
import re

def add_klifs_info(df):
    klifs = pd.read_csv('../../source_data/16_KLIFS/KLIFS_compiled.csv')

    in_klifs_list = []
    for i, val in df.iterrows():
        hgvsp = val['Top_HGVSp']
        gene = val['gene']
        klifs_idx = list(klifs[gene])

        in_klifs = False

        
        pattern = r"-?\d*\.?\d+" 
        matches = re.findall(pattern, hgvsp)

        for i in matches:
            if int(i) in klifs_idx:
                in_klifs = True
                break

        in_klifs_list.append(in_klifs)
                                   
    df['KLIFS_residue'] = in_klifs_list

    return df

def generate_edit_info(m_abe, ABE_canonical_edits, ABE_non_canonical_edits):

    m_abe_filtered = m_abe[m_abe['classification']=='targeting']
    #m_cbe_filtered = m_cbe[m_cbe['classification']=='targeting']

    abe_guides = m_abe_filtered['gRNA_id'].to_list()
    #cbe_guides = m_cbe_filtered['gRNA_id'].to_list()

    ABE_edit_info = pd.DataFrame({'gRNA_id':abe_guides})
    ABE_edit_info['gene'] = [x.split('_')[1] for x in ABE_edit_info['gRNA_id']]
    #CBE_edit_info = pd.DataFrame({'gRNA_id':cbe_guides})

    for i, val in ABE_edit_info.iterrows():
        guide = val['gRNA_id']

        subset = ABE_canonical_edits[ABE_canonical_edits['gRNA_id']==guide]
        subset_hgvsp = subset[['HGVSp', '#Reads']].groupby('HGVSp').sum().reset_index().sort_values('#Reads', ascending=False)
        subset_hgvsp['%Reads'] = subset_hgvsp['#Reads']/subset_hgvsp['#Reads'].sum()*100
        subset_hgvsp = subset_hgvsp.sort_values('%Reads', ascending=False).reset_index(drop=True)

        try:
            non_wt = subset_hgvsp[subset_hgvsp['HGVSp']!='WT'].sort_values('%Reads', ascending=False).reset_index(drop=True)

            if len(non_wt)>0:
                ABE_edit_info.loc[ABE_edit_info['gRNA_id']==guide, 'Top_HGVSp'] = non_wt.loc[0, 'HGVSp']

                pattern = r"-?\d*\.?\d+" 
                # The expression \d+\.\d+|\d+ matches both floating-point numbers and integers

                matches = re.findall(pattern, non_wt.loc[0, 'HGVSp'])
                if len(matches)>0:
                    ABE_edit_info.loc[ABE_edit_info['gRNA_id']==guide, 'Top_HGVSp_codon'] = int(matches[0])
                else:
                    ABE_edit_info.loc[ABE_edit_info['gRNA_id']==guide, 'Top_HGVSp_codon'] = int(0)

                ABE_edit_info.loc[ABE_edit_info['gRNA_id']==guide, 'Top_HGVSp_%Reads_adjusted'] = non_wt.loc[0, '%Reads']

            elif len(non_wt)==0:
                ABE_edit_info.loc[ABE_edit_info['gRNA_id']==guide, 'Top_HGVSp'] = 'WT'
                ABE_edit_info.loc[ABE_edit_info['gRNA_id']==guide, 'Top_HGVSp_codon'] = 0
                ABE_edit_info.loc[ABE_edit_info['gRNA_id']==guide, 'Top_HGVSp_%Reads_adjusted'] = subset_hgvsp[subset_hgvsp['HGVSp']=='WT']['%Reads'].values[0]
            
            if len(subset_hgvsp[subset_hgvsp['HGVSp']=='WT'])>0:
                ABE_edit_info.loc[ABE_edit_info['gRNA_id']==guide, 'WT_%Reads_adjusted'] = subset_hgvsp[subset_hgvsp['HGVSp']=='WT']['%Reads'].values[0]
            else:
                ABE_edit_info.loc[ABE_edit_info['gRNA_id']==guide, 'WT_%Reads_adjusted'] = 0

            subset_noncanonical = ABE_non_canonical_edits[ABE_non_canonical_edits['gRNA_id']==guide]

            canonical_reads = subset_hgvsp['#Reads'].sum()
            noncanonical_reads = subset_noncanonical['#Reads'].sum()

            ABE_edit_info.loc[ABE_edit_info['gRNA_id']==guide, 'Canonical_Reads'] = canonical_reads
            #ABE_edit_info.loc[ABE_edit_info['gRNA_id']==guide, 'Noncanonical_Reads'] = noncanonical_reads

            ABE_edit_info.loc[ABE_edit_info['gRNA_id']==guide, 'NonCanonical_%Reads'] = noncanonical_reads/(canonical_reads+noncanonical_reads)*100

        except:
            print(f'Error processing guide {guide}')

            continue

    ABE_edit_info = add_klifs_info(ABE_edit_info)

    return ABE_edit_info





In [284]:
LFC_cutoff = 2
FDR_cutoff = 0.05

m_abe, m_cbe = generate_top_table(ABE_LFC, CBE_LFC, conditions_comb, LFC_cutoff, FDR_cutoff)

print(len(m_abe))
print(len(m_cbe))

136
54


In [285]:
ABE_edit_info = generate_edit_info(m_abe, ABE_canonical_edits, ABE_non_canonical_edits)
CBE_edit_info = generate_edit_info(m_cbe, CBE_canonical_edits, CBE_non_canonical_edits)

Error processing guide gRNA_CDK19_targ_6662
Error processing guide gRNA_CDK19_targ_6661


In [ ]:
#---merge
abe_merged = pd.merge(ABE_edit_info, m_abe, on='gRNA_id', how='outer')
cbe_merged = pd.merge(CBE_edit_info, m_cbe, on='gRNA_id', how='outer')

#---filtration
editing_cutoff = 20
base_min = 100
noncanonical_perc_max = 50

abe_merged_filtered = abe_merged[(abe_merged['classification']=='targeting') & (abe_merged['Top_HGVSp']!='WT')]
cbe_merged_filtered = cbe_merged[(cbe_merged['classification']=='targeting') & (cbe_merged['Top_HGVSp']!='WT')]

cbe_merged_filtered = cbe_merged_filtered[(cbe_merged_filtered['Top_HGVSp_%Reads_adjusted']>editing_cutoff) & (cbe_merged_filtered['NonCanonical_%Reads']<noncanonical_perc_max) & (cbe_merged_filtered['base_RAW_DMSO']>base_min)].sort_values(by='gene')
abe_merged_filtered = abe_merged_filtered[(abe_merged_filtered['Top_HGVSp_%Reads_adjusted']>editing_cutoff) & (abe_merged_filtered['NonCanonical_%Reads']<noncanonical_perc_max) & (abe_merged_filtered['base_RAW_DMSO']>base_min)].sort_values(by='gene')

print(len(abe_merged_filtered))
print(len(cbe_merged_filtered))

75
28


In [295]:

abe_merged_filtered.to_excel('ABE_merged_filtered.xlsx', index=False)
cbe_merged_filtered.to_excel('CBE_merged_filtered.xlsx', index=False)

# get truncating mutations


In [211]:
ters_CBE = [i for i in CBE_canonical_edits['HGVSp'].unique() if '*' in i]

ters_ABE = [i for i in ABE_canonical_edits['HGVSp'].unique() if '*' in i]

ter = CBE_canonical_edits[CBE_canonical_edits['HGVSp'].isin(ters_CBE)].sort_values(by='%Reads', ascending=False).reset_index(drop=True)

genes = list(CBE_canonical_edits['Gene'].unique())

toph = []
for gene in genes:
    subset = ter[ter['Gene']==gene].sort_values(by='%Reads', ascending=False).drop_duplicates(subset=['gRNA_id']).reset_index(drop=True)
    topx = subset.head(3)
    toph.append(topx)

pd.concat(toph)

,Edit,HGVSp,Num_edits,DNA Change,Canonical_edit,Canonical_window,gRNA_id,#Reads,%Reads,Editor,Gene
0,TGAGATTGACTAGCTCTTCC,Q211*,1,"+11C>T,",True,False,gRNA_CDK2_targ_7123,4731,47.083997,CBE,CDK2
1,CGGGCTTACTTGGGGAAACT,W243*,2,"+6C>T,+7C>T,",True,True,gRNA_CDK2_targ_7392,2768,45.354744,CBE,CDK2
2,GGGCTTACTTGGGGAAACTT,W243*,2,"+5C>T,+6C>T,",True,True,gRNA_CDK2_targ_7391,1152,43.553875,CBE,CDK2
0,CTGAGGTGACTGGAGGCTTT,R62*,1,"+7C>T,",True,True,gRNA_CDK4_targ_7911,17299,70.709176,CBE,CDK4
1,GACTAGTTGGGCAAAATCTT,Q222*,1,"+4C>T,",True,True,gRNA_CDK4_targ_7774,5783,50.374564,CBE,CDK4
2,TCACTGAGATCTGAAGCCAG,R139*,1,"+5C>T,",True,True,gRNA_CDK4_targ_7842,42530,46.863981,CBE,CDK4
0,TGATTAACTAGGAAAAATCT,Q227*,1,"+5C>T,",True,True,gRNA_CDK6_targ_8285,659,57.959543,CBE,CDK6
1,AGGTTAGTTTTCTTCTCCTG,D242N_W243*,3,"+4C>T,+5C>T,+9C>T,",True,False,gRNA_CDK6_targ_8043,4128,50.787402,CBE,CDK6
2,CATACTTTTAGGACCTGGAA,Q301*,2,"+8C>T,+9C>T,",True,False,gRNA_CDK6_targ_8240,6682,46.528793,CBE,CDK6
0,AGGTGTAGGTAGCCAAAAGC,A164V_Q165*,2,"+4C>T,+6C>T,",True,True,gRNA_CDK12_targ_1880,4421,79.844681,CBE,CDK12


In [207]:
genes = list(CBE_canonical_edits['Gene'].unique())
genes

['CDK2', 'CDK4', 'CDK6', 'CDK12', 'CDK13', 'CDK19', 'CDK7', 'CDK8', 'CDK9']